# MaxPool2d

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mitchell-Mirano/sorix/blob/develop/docs/learn/layers/09-MaxPool2d.ipynb)
[![Open in GitHub](https://img.shields.io/badge/Open%20in-GitHub-black?logo=github)](https://github.com/Mitchell-Mirano/sorix/blob/develop/docs/learn/layers/09-MaxPool2d.ipynb)
[![Open in Docs](https://img.shields.io/badge/Open%20in-Docs-blue?logo=readthedocs)](http://127.0.0.1:8000/sorix/learn/layers/09-MaxPool2d)

The **MaxPool2d** layer applies a 2D max pooling operation over an input signal. It is a non-linear downsampling mechanism deeply utilized in Convolutional Neural Networks to reduce the spatial resolution of feature maps, thereby decreasing the computational cost and controlling overfitting.

## Mathematical definition

Given an input tensor $\mathbf{X} \in \mathbb{R}^{N \times C \times H \times W}$, the layer extracts windows of size $K_H \times K_W$ and computes the maximum value within each mapped sub-region.

For a single coordinate in the output tensor $\mathbf{Y} \in \mathbb{R}^{N \times C \times H_{out} \times W_{out}}$, the calculation is strictly defined as:

$$
Y_{n, c, h, w} = \max_{k_h=0}^{K_H-1} \max_{k_w=0}^{K_W-1} \mathbf{X}_{n, c, h \times s_h + k_h, w \times s_w + k_w}
$$

where:
- $s_h, s_w$ is the **stride** factor (usually equal to the kernel size to avoid overlap handling).
 
### Output Dimensions

Similarly to convolutions, the new spatial shapes collapse based on the window projection:
$$
H_{out} = \left\lfloor \frac{H - K_H}{s_h} + 1 \right\rfloor \quad \text{and} \quad W_{out} = \left\lfloor \frac{W - K_W}{s_w} + 1 \right\rfloor
$$

## Interpretation and Explainability

Max Pooling provides form of **translation invariance**. By taking the maximum value inside a given local region, the neural network extracts the "most prominent" activated feature (e.g., the sharpest outline or the brightest pixel map activation) and ignores its exact microscopic coordinate. This prevents the network from memorizing pixel-perfect coordinates and forces it to understand generalized objects.

Additionally, standard Pooling (`kernel_size=2`, `stride=2`) scales down the image sizes by exactly half ($1/2$), aggressively shrinking parameter requirement in consecutive `Linear` headers.

## Gradient Routing (Backpropagation)

Pooling layers possess **zero trainable parameters**; however, they actively participate in automatic differentiation (`Autograd`).
During the backward pass ($\frac{\partial \mathcal{L}}{\partial \mathbf{X}}$), the gradients $\partial \mathbf{Y}$ only propagate back through the explicit item that scored the `maximum` value. The elements that were "ignored" during $max$ receive a gradient of $0$.
Sorix implements this via dense spatial `argmax` masking buffers, dynamically associating forward indices with backward derivatives.

In [ ]:
from sorix import tensor
from sorix.nn import MaxPool2d
import numpy as np

In [ ]:
# Create a tensor mimicking intermediate CNN feature maps:
N, C, H, W = 1, 16, 28, 28  # 1 image, 16 feature maps, 28x28 size
X = tensor(np.random.randn(N, C, H, W).astype(np.float32))

print("Input features shape:", X.shape)

In [ ]:
# Instantiate a MaxPool2d layer to cut dimensions in half
pool = MaxPool2d(kernel_size=2, stride=2)

# Forward pass
Y = pool(X)

print("Output dimension cut in half:", Y.shape)